# 04 - Extract NOAA Atlas 14 precipitation frequency (NWS PFDS)

Depth-duration-frequency estimates for the config-driven basin points, via the PFDS
point CSV API. Raw responses land in `data/raw/atlas14/`; a tidy combined table goes to
`data/processed/atlas14_ddf.csv`. CA points resolve to Atlas 14 Volume 6, NV points to
Volume 1 - the API handles both transparently.

In [ ]:
import sys
print("Python:", sys.executable)

import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

# Pipeline root = climate/ (parent of notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.io import load_config, get_logger, append_manifest, sha256_file

cfg = load_config()
log = get_logger("04_extract_atlas14")

RAW = ROOT / cfg["paths"]["raw"]
PROCESSED = ROOT / cfg["paths"]["processed"]
OUTPUTS = ROOT / cfg["paths"]["outputs"]
for p in (RAW, PROCESSED, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

BBOX = cfg["study_area"]["bbox"]
log.info(f"bbox: lon {BBOX['lon_min']}..{BBOX['lon_max']}, lat {BBOX['lat_min']}..{BBOX['lat_max']}")

In [ ]:
import io as pyio
import requests

A = cfg["sources"]["atlas14"]
a14_dir = RAW / "atlas14"
a14_dir.mkdir(exist_ok=True)

DURATIONS = ["5-min", "10-min", "15-min", "30-min", "60-min", "2-hr", "3-hr", "6-hr",
             "12-hr", "24-hr", "2-day", "3-day", "4-day", "7-day", "10-day", "20-day",
             "30-day", "45-day", "60-day"]
ARIS = [1, 2, 5, 10, 25, 50, 100, 200, 500, 1000]

rows = []
for pt in A["points"]:
    out = a14_dir / f"pfds_{pt['name']}.csv"
    if not (out.exists() and not cfg["run"]["overwrite_downloads"]):
        r = requests.get(A["pfds_url"], params={"lat": pt["lat"], "lon": pt["lon"],
                                                "data": "depth", "units": A["units"],
                                                "series": A["series"]}, timeout=60)
        r.raise_for_status()
        out.write_text(r.text)
        append_manifest({"file": str(out.relative_to(ROOT)), "source_url": r.url,
                         "size_bytes": out.stat().st_size, "sha256": sha256_file(out),
                         "retrieved_date": str(pd.Timestamp.today().date()),
                         "notebook": "04_extract_atlas14"})
        log.info(f"fetched {pt['name']}")
    # parse: quantile block lines look like  "24-hr:, 1.2, 1.5, ..."
    txt = out.read_text()
    started = False
    for line in txt.splitlines():
        parts = [p.strip().rstrip(":") for p in line.split(",")]
        if parts and parts[0] in DURATIONS:
            started = True
            vals = parts[1:1 + len(ARIS)]
            for ari, v in zip(ARIS, vals):
                try:
                    depth_in = float(v)
                except (TypeError, ValueError):
                    continue
                rows.append({"point": pt["name"], "lat": pt["lat"], "lon": pt["lon"],
                             "duration": parts[0], "ari_years": ari,
                             "depth_in": depth_in, "depth_mm": round(depth_in * 25.4, 2)})
        elif started and parts and parts[0] not in DURATIONS:
            started = False

ddf = pd.DataFrame(rows)
ddf.to_csv(PROCESSED / "atlas14_ddf.csv", index=False)
log.info(f"tidy DDF table: {len(ddf)} rows, {ddf['point'].nunique() if len(ddf) else 0} points "
         f"-> data/processed/atlas14_ddf.csv")
ddf[ddf["duration"].eq("24-hr") & ddf["ari_years"].isin([2, 25, 100])] if len(ddf) else ddf

## NOAA Atlas 15 stub
Atlas 15 introduces non-stationary (climate-adjusted) precipitation frequency. As of
mid-2026 it is **pilot only** (Montana pilot published; national rollout phased). When
CA/NV coverage publishes, replace the Atlas 14 point pulls above with Atlas 15 DDF grids
and add the future-scenario adjustment factors to the transform notebook.

Status check: https://www.weather.gov/owp/hdsc_atlas15

In [ ]:
log.info(f"Atlas 15 status URL (check before Workshop 1): "
         f"{A['atlas15']['status_url']} - {A['atlas15']['note']}")